## 1. Install Dependencies

Run this once if your environment does not already have these packages.

In [1]:
import subprocess
import sys
from pathlib import Path

requirements = Path("requirements.txt")
if not requirements.exists():
    requirements = Path("llm/api/requirements.txt")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements)])

0

## 2. Load Environment and Model

If your Hugging Face repo is private, set `HF_TOKEN` in `.env`. If it is public, the token is optional.

In [2]:
import math
import os
import re
from collections import Counter

import torch
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_huggingface import HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from peft import PeftModel
from pypdf import PdfReader
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

load_dotenv(".env", override=True)

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_REPO = "Glccampos/llm_qween"
HF_TOKEN = os.getenv("HF_TOKEN")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Device: {DEVICE}")
print(f"Adapter repo: https://huggingface.co/{ADAPTER_REPO}")

Device: cpu
Adapter repo: https://huggingface.co/Glccampos/llm_qween


In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_REPO,
    token=HF_TOKEN,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    token=HF_TOKEN,
    dtype=DTYPE,
    trust_remote_code=True,
)
base_model.to(DEVICE)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_REPO,
    token=HF_TOKEN,
)
model.eval()

print("Loaded base model and LoRA adapter.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded base model and LoRA adapter.


## 3. Wrap the Model with LangChain

The adapter was trained with chat-style prompts, so we format each request with Qwen's chat template before passing it to the text-generation pipeline.

In [4]:
USE_SAMPLING = True
TEMPERATURE = 0.1
TOP_P = 0.9
TOP_K = 50

# Temperature/top-p/top-k only make sense when sampling is enabled.
generation_kwargs = {
    "do_sample": USE_SAMPLING,
    "max_new_tokens": 128,
    "pad_token_id": tokenizer.eos_token_id,
    "eos_token_id": tokenizer.eos_token_id,
}

if USE_SAMPLING:
    generation_kwargs.update(
        {
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "top_k": TOP_K,
        }
    )
else:
    # Neutral defaults keep deterministic decoding free of sampling warnings.
    generation_kwargs.update(
        {
            "temperature": 0.1,
            "top_p": 1.0,
            "top_k": 50,
        }
    )

model.generation_config.update(**generation_kwargs)
model.generation_config.max_length = None

text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    clean_up_tokenization_spaces=False,
    device=0 if DEVICE == "cuda" else -1,
)
text_generation_pipeline.generation_config.update(**generation_kwargs)
text_generation_pipeline.generation_config.max_length = None

llm = HuggingFacePipeline(
    pipeline=text_generation_pipeline,
    pipeline_kwargs=generation_kwargs,
)


def format_qa_prompt(inputs):
    context = inputs.get("context", "")
    question = inputs["question"]
    user_prompt = (
        "Answer the question using only the context when context is provided.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:\n{question}"
    )
    messages = [
        {"role": "system", "content": "You are a helpful Q&A assistant."},
        {"role": "user", "content": user_prompt},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


qa_chain = RunnableLambda(format_qa_prompt) | llm | StrOutputParser()

print("LangChain Q&A chain is ready.")

LangChain Q&A chain is ready.


## 4. Build a RAG Chain from the PDF

In [5]:
import math
import re
from collections import Counter

from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

PDF_PATH = "Analytics Engineer .pdf"

reader = PdfReader(PDF_PATH)
pdf_text = "\n".join(page.extract_text() or "" for page in reader.pages)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
)
chunks = text_splitter.split_text(pdf_text)


def tokenize_for_retrieval(text):
    return re.findall(r"[a-zA-Z0-9]+", text.lower())


chunk_term_counts = [Counter(tokenize_for_retrieval(chunk)) for chunk in chunks]
document_frequencies = Counter(
    term for counts in chunk_term_counts for term in counts.keys()
)


def retrieve_context(question, top_k=3):
    query_terms = tokenize_for_retrieval(question)
    if not query_terms:
        return "\n\n".join(chunks[:top_k])

    scored_chunks = []
    for index, (chunk, term_counts) in enumerate(zip(chunks, chunk_term_counts)):
        score = 0.0
        for term in query_terms:
            if term not in term_counts:
                continue
            idf = math.log((len(chunks) + 1) / (document_frequencies[term] + 1)) + 1
            score += term_counts[term] * idf
        scored_chunks.append((score, index, chunk))

    best_chunks = [
        chunk
        for score, _, chunk in sorted(scored_chunks, reverse=True)[:top_k]
        if score > 0
    ]
    return "\n\n---\n\n".join(best_chunks or chunks[:top_k])


def answer_from_pdf(question):
    context = retrieve_context(question)
    return qa_chain.invoke({"context": context, "question": question}).strip()


print(f"Loaded {len(chunks)} chunk(s) from {PDF_PATH}")

Loaded 4 chunk(s) from Analytics Engineer .pdf


## 5. Ask Questions About the PDF

In [6]:
question = "What skills are required for the Analytics Engineer role?"
answer = answer_from_pdf(question)
print(answer)

Hands-on experience with dbt, ETL optimization, version control tools such as GitHub or GitLab, and visualization tools such as Sigma or Power BI.


## 6. Connect LangChain to the Local Trino MCP Server

This starts your local `mcp/trino_mcp.py` server over stdio and loads its tools into LangChain. The MCP server uses the `.env` file in `/mnt/c/Users/guslc/project/mcp` for the Trino connection settings.

In [12]:
from pathlib import Path

from langchain_mcp_adapters.client import MultiServerMCPClient

TRINO_MCP_DIR = Path("/mnt/c/Users/guslc/project/mcp")
TRINO_MCP_SERVER = TRINO_MCP_DIR / "trino_mcp.py"
TRINO_MCP_PYTHON = TRINO_MCP_DIR / ".venv" / "bin" / "python"

if not TRINO_MCP_SERVER.exists():
    raise FileNotFoundError(f"MCP server file not found: {TRINO_MCP_SERVER}")
if not TRINO_MCP_PYTHON.exists():
    raise FileNotFoundError(
        f"MCP Python interpreter not found: {TRINO_MCP_PYTHON}. Run `uv sync` in {TRINO_MCP_DIR}."
    )

mcp_client = MultiServerMCPClient(
    {
        "trino": {
            "transport": "stdio",
            "command": str(TRINO_MCP_PYTHON),
            "args": [str(TRINO_MCP_SERVER)],
            "cwd": str(TRINO_MCP_DIR),
        }
    }
)

trino_tools = await mcp_client.get_tools()
trino_tool_names = [tool.name for tool in trino_tools]
trino_tool_names

['query', 'list_catalogs', 'list_schemas', 'list_tables', 'describe_table']

In [13]:
import json


def _normalize_mcp_result(value):
    """Unwrap LangChain MCP tool output into the dict returned by trino_mcp.py."""
    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return {"text": value}

    if isinstance(value, list):
        if len(value) == 1:
            item = value[0]
            if isinstance(item, dict) and "text" in item:
                return _normalize_mcp_result(item["text"])
            return _normalize_mcp_result(item)
        return {"rows": value, "row_count": len(value)}

    return {"value": value}


async def call_trino_tool(name: str, arguments: dict | None = None):
    """Call a Trino MCP tool by name from the loaded LangChain tools."""
    arguments = arguments or {}
    matches = [tool for tool in trino_tools if tool.name == name or tool.name.endswith(name)]
    if not matches:
        raise ValueError(f"Tool {name!r} not found. Available tools: {trino_tool_names}")
    raw_result = await matches[0].ainvoke(arguments)
    return _normalize_mcp_result(raw_result)


catalogs = await call_trino_tool("list_catalogs")
catalogs

{'columns': ['Catalog'],
 'rows': [['clickhouse'], ['iceberg'], ['system'], ['tpcds'], ['tpch']],
 'row_count': 5,
 'max_rows': 1000}

In [14]:
schemas = await call_trino_tool("list_schemas", {"catalog": "iceberg"})
schemas

{'columns': ['Schema'],
 'rows': [['default'], ['forecast'], ['information_schema']],
 'row_count': 3,
 'max_rows': 1000}

## 7. Ask Trino Questions with Natural Language

`ask_trino(...)` uses the MCP tools to inspect the selected Trino schema, asks the LangChain model to write SQL, executes that SQL through the MCP `query` tool, and summarizes the result.

In [15]:
import re


def _rows_to_markdown(columns, rows, max_rows=20):
    rows = rows[:max_rows]
    if not columns:
        return str(rows)
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = ["| " + " | ".join(str(value) for value in row) + " |" for row in rows]
    return "\n".join([header, separator, *body])    


def _extract_sql(model_output: str) -> str:
    text = model_output.strip()
    fenced = re.search(r"```(?:sql)?\s*(.*?)```", text, flags=re.IGNORECASE | re.DOTALL)
    if fenced:
        text = fenced.group(1).strip()
    text = re.sub(r"^SQL\s*:\s*", "", text, flags=re.IGNORECASE).strip()
    if len(text) >= 2 and text[0] == text[-1] and text[0] in {"'", '"'}:
        text = text[1:-1].strip()
    text = text.rstrip(";").strip()

    lowered = text.lower()
    allowed_prefixes = ("select", "with", "show", "describe", "desc")
    forbidden_words = ("insert", "update", "delete", "drop", "alter", "create", "truncate", "merge")
    if not lowered.startswith(allowed_prefixes):
        raise ValueError(f"Model did not produce a read-only SQL query: {model_output!r}")
    if any(re.search(rf"\b{word}\b", lowered) for word in forbidden_words):
        raise ValueError(f"Refusing to execute non-read-only SQL: {text}")
    return text


async def get_trino_schema_context(catalog="iceberg", schema="forecast", max_tables=20):
    table_result = await call_trino_tool("list_tables", {"schema": f"{catalog}.{schema}"})
    table_names = [row[0] for row in table_result.get("rows", [])]

    table_descriptions = []
    for table_name in table_names[:max_tables]:
        full_table_name = f"{catalog}.{schema}.{table_name}"
        description = await call_trino_tool("describe_table", {"table": full_table_name})
        columns = []
        for row in description.get("rows", []):
            if len(row) >= 2 and row[0]:
                columns.append(f"{row[0]} {row[1]}")
        table_descriptions.append(f"{full_table_name}: " + ", ".join(columns))

    return "\n".join(table_descriptions)


async def generate_trino_sql(question: str, catalog="iceberg", schema="forecast"):
    schema_context = await get_trino_schema_context(catalog=catalog, schema=schema)
    messages = [
        {
            "role": "system",
            "content": (
                "You write read-only Trino SQL. Return exactly one SQL query and no markdown. "
                "Use fully qualified table names from the provided schema context."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Schema context:\n{schema_context}\n\n"
                f"Question: {question}\n\n"
                "Write the Trino SQL query."
            ),
        },
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return _extract_sql(llm.invoke(prompt))


async def ask_trino(question: str, catalog="iceberg", schema="forecast", max_rows=100):
    sql = await generate_trino_sql(question, catalog=catalog, schema=schema)
    result = await call_trino_tool("query", {"sql": sql, "max_rows": max_rows})

    preview = _rows_to_markdown(result.get("columns", []), result.get("rows", []))
    messages = [
        {"role": "system", "content": "Summarize Trino query results clearly and briefly."},
        {
            "role": "user",
            "content": (
                f"Question: {question}\n"
                f"SQL: {sql}\n\n"
                f"Result preview:\n{preview}\n\n"
                "Answer the question using the SQL result."
            ),
        },
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    answer = llm.invoke(prompt).strip()
    return {"question": question, "sql": sql, "result": result, "answer": answer}

In [19]:
response = await ask_trino(
    "How many prediction have the probability > 0.8",
    # catalog="iceberg",
    # schema="forecast",
)

print(response["sql"])
print(response["answer"])
response["result"]

SELECT COUNT(event_id)
FROM iceberg.forecast.prediction_events
WHERE probability > 0.8
The answer is that there are 29,2720 events with a probability greater than 0.8.


{'columns': ['_col0'], 'rows': [[292720]], 'row_count': 1, 'max_rows': 100}